### Read RA list

In [13]:
# pip install pandas
# pip install openpyxl

In [2]:
import pandas as pd
import numpy as np

ra_list = pd.read_excel('D:/SVL/2021-06-16_ra_list_v1.6.xlsx', keep_default_na = False, na_values = [''])
ra_list.head()


,Registration Authority Code,Country,Country Code,Jurisdiction (country or region),International name of Register,Local name of Register,International name of organisation responsible for the Register,Local name of organisation responsible for the Register,Website,Comments
0,RA777777,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,to be used for 'Public Legal Documents' for Ge...
1,RA888888,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,temporary code; new registration authority cod...
2,RA999999,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,no registration authority available for this l...
3,RA000001,Afghanistan,AF,Afghanistan,Afghanistan Central Business Registry (Ministr...,NaN,Afghanistan Central Business Registry (Ministr...,NaN,http://www.acbr.gov.af/,NaN
4,RA000759,Afghanistan,AF,Afghanistan,Register of Licensed Financial Institutions,NaN,Da Afghanistan Bank,NaN,https://dab.gov.af/index.php/Licensed-Finacial...,to be used for banks and some other licensed f...


### Finding anomalies
#### define a function that finds anomalies in the dataset 

In [2]:
def find_anomalies(raList, leiData):
    
    jurisdiction_by_ra = raList.fillna('--').groupby('Registration Authority Code')['Country Code'].apply(list)
#     print('Jurisdiction by RA given as a list: ', jurisdiction_by_ra.head(15))
    
    tmp=leiData[leiData['Entity.LegalJurisdiction'].notnull() & 
       leiData['Entity.RegistrationAuthority.RegistrationAuthorityID'].notnull() &
       (~leiData['Entity.RegistrationAuthority.RegistrationAuthorityID'].isin(['RA777777', 'RA888888', 'RA999999']))
                
    ].assign(ra_jurisdictions=lambda d: d['Entity.RegistrationAuthority.RegistrationAuthorityID'].apply(lambda x: tuple(jurisdiction_by_ra.get(x, [])))
    ).assign(not_covered= lambda d: d.apply(lambda row: row['Entity.LegalJurisdiction'][:2] not in jurisdiction_by_ra.get(row['Entity.RegistrationAuthority.RegistrationAuthorityID'], []), axis=1)) 
    
    not_covered = tmp[tmp.not_covered]
    
    return not_covered

### Chunking LEI data, Call the function to find anomalies, Concatenate the results

In [3]:
# chunks = pd.concat(lei_data, ignore_index = True)


cols = ['LEI', 'Entity.LegalName','Entity.RegistrationAuthority.RegistrationAuthorityID', 'Entity.RegistrationAuthority.RegistrationAuthorityEntityID', 
        'Entity.RegistrationAuthority.OtherRegistrationAuthorityID','Entity.LegalJurisdiction','Entity.EntityStatus','Registration.RegistrationStatus', 'Registration.ManagingLOU']

# num_rows = 1

# num_rows = 0
chunks = pd.DataFrame()
# Step 1: using chunksize method in order to reduce memory consumption
lei_data = pd.read_csv('D:/SVL/20211202-0800-gleif-goldencopy-lei2-golden-copy.csv',
    low_memory=False, dtype=str, na_values=[''], keep_default_na=False, usecols = cols, chunksize=400000)
for chunk in lei_data:
    # calculate the size of total data/ total number of rows
#     num_rows = num_rows + len(chunk)
#     print('CHUNK LEIs')
#     print(chunk)

    # Step 2: Call the function. Show anomalies
    result = find_anomalies (ra_list, chunk)
    # print('ANOMALIES', result)
    
# print(num_rows) 

    # chunking into smaller csv files
#     chunk.to_csv('D:/SVL/Chunks/Lei golden copy chunk' + str(num_rows) + '.csv', index= False)
#     num_rows += 1
    
    # Step 3: Concatenate the results
    chunks = pd.concat([chunks, result]) 
new_chunks = chunks.reset_index()
new_chunks
# len(chunks) 
# chunk.info()

,index,LEI,Entity.LegalName,Entity.RegistrationAuthority.RegistrationAuthorityID,Entity.RegistrationAuthority.OtherRegistrationAuthorityID,Entity.RegistrationAuthority.RegistrationAuthorityEntityID,Entity.LegalJurisdiction,Entity.EntityStatus,Registration.RegistrationStatus,Registration.ManagingLOU,ra_jurisdictions,not_covered
0,12037,2138002FA7TJLH9VCN93,NDM LIMITED,RA000161,NaN,ΑΕ 3085,KY,ACTIVE,ISSUED,213800WAVVOPS85N2205,"(CY,)",True
1,47581,2138008NLWEZC6R7KJ57,IGNIS FUNDS SICAV - IGNIS ASIA PACIFIC FUND,RA000404,NaN,612291,LU,INACTIVE,RETIRED,213800WAVVOPS85N2205,"(IE,)",True
2,52192,2138009H2TBPW7JVN539,CIENEGA HOLDING B.V.,RA000443,NaN,OC 419,NL,INACTIVE,RETIRED,213800WAVVOPS85N2205,"(MT,)",True
3,65214,213800BRGVDAR6VH3561,IGNIS FUNDS SICAV - IGNIS CHINA FUND,RA000404,NaN,612991,LU,INACTIVE,RETIRED,213800WAVVOPS85N2205,"(IE,)",True
4,70669,213800CPQOTAW6TZOY27,HENDERSON HORIZON FUND - WORLD SELECT FUND,RA000592,NaN,144111,LU,INACTIVE,RETIRED,213800WAVVOPS85N2205,"(GB,)",True
...,...,...,...,...,...,...,...,...,...,...,...,...
80,1988944,984500G768A11O365727,EIFFAGE INFRAESTRUCTURAS SA SPÓŁKA AKCYJNA ODD...,RA000484,NaN,0000833468,ES,ACTIVE,ISSUED,529900T8BM49AURSDO55,"(PL,)",True
81,1989635,984500I13E8FF4CD7C96,TEKNOVEKST,RA000472,NaN,995 476 711,GB,ACTIVE,ISSUED,529900T8BM49AURSDO55,"(NO,)",True
82,1993130,984500R2B00E5EE0DC75,Kotak Mahindra Bank Limited,RA000660,NaN,3338,IN,ACTIVE,ISSUED,529900T8BM49AURSDO55,"(AE,)",True
83,1994416,984500UAXGE4D2451072,EVIL EMPIRE NORGE LIMITED,RA000472,NaN,991 223 436,GB,ACTIVE,ISSUED,529900T8BM49AURSDO55,"(NO,)",True


In [173]:
data_merged = pd.merge(new_chunks, ra_list, left_on = 'Entity.RegistrationAuthority.RegistrationAuthorityID', right_on='Registration Authority Code', how='left')
data_merged


,index,LEI,Entity.LegalName,Entity.RegistrationAuthority.RegistrationAuthorityID,Entity.RegistrationAuthority.OtherRegistrationAuthorityID,Entity.RegistrationAuthority.RegistrationAuthorityEntityID,Entity.LegalJurisdiction,Entity.EntityStatus,Registration.RegistrationStatus,Registration.ManagingLOU,...,Registration Authority Code,Country,Country Code,Jurisdiction (country or region),International name of Register,Local name of Register,International name of organisation responsible for the Register,Local name of organisation responsible for the Register,Website,Comments
0,12037,2138002FA7TJLH9VCN93,NDM LIMITED,RA000161,NaN,ΑΕ 3085,KY,ACTIVE,ISSUED,213800WAVVOPS85N2205,...,RA000161,Cyprus,CY,Cyprus,Companies Section,NaN,Department of Registrar of Companies and Offic...,NaN,http://www.mcit.gov.cy/mcit/drcor/drcor.nsf/in...,NaN
1,47581,2138008NLWEZC6R7KJ57,IGNIS FUNDS SICAV - IGNIS ASIA PACIFIC FUND,RA000404,NaN,612291,LU,INACTIVE,RETIRED,213800WAVVOPS85N2205,...,RA000404,Ireland,IE,Ireland,Registers for financial service providers and ...,NaN,Central Bank of Ireland,NaN,registers.centralbank.ie,NaN
2,52192,2138009H2TBPW7JVN539,CIENEGA HOLDING B.V.,RA000443,NaN,OC 419,NL,INACTIVE,RETIRED,213800WAVVOPS85N2205,...,RA000443,Malta,MT,Malta,Registry of Companies,Registru tal-Kumpaniji ta'Malta,Malta Financial Services Authority - Registry ...,NaN,https://registry.mbr.mt/ROC/index.jsp,NaN
3,65214,213800BRGVDAR6VH3561,IGNIS FUNDS SICAV - IGNIS CHINA FUND,RA000404,NaN,612991,LU,INACTIVE,RETIRED,213800WAVVOPS85N2205,...,RA000404,Ireland,IE,Ireland,Registers for financial service providers and ...,NaN,Central Bank of Ireland,NaN,registers.centralbank.ie,NaN
4,70669,213800CPQOTAW6TZOY27,HENDERSON HORIZON FUND - WORLD SELECT FUND,RA000592,NaN,144111,LU,INACTIVE,RETIRED,213800WAVVOPS85N2205,...,RA000592,United Kingdom,GB,United Kingdom,Financial Services Register,Financial Services Register,Financial Conduct Authority,Financial Conduct Authority,https://register.fca.org.uk,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113,1988944,984500G768A11O365727,EIFFAGE INFRAESTRUCTURAS SA SPÓŁKA AKCYJNA ODD...,RA000484,NaN,0000833468,ES,ACTIVE,ISSUED,529900T8BM49AURSDO55,...,RA000484,Poland,PL,Poland,National Court Register,Krajowy Rejestr Sądowy (KRS),Ministry of Justice,Minsterstwo Sprawiedliwosci,https://ekrs.ms.gov.pl/web/wyszukiwarka-krs/st...,NaN
114,1989635,984500I13E8FF4CD7C96,TEKNOVEKST,RA000472,NaN,995 476 711,GB,ACTIVE,ISSUED,529900T8BM49AURSDO55,...,RA000472,Norway,NO,Norway,The Register of Business Enterprises,Foretaksregisteret,The Brønnøysund Register Centre,NaN,https://www.brreg.no/,NaN
115,1993130,984500R2B00E5EE0DC75,Kotak Mahindra Bank Limited,RA000660,NaN,3338,IN,ACTIVE,ISSUED,529900T8BM49AURSDO55,...,RA000660,United Arab Emirates,AE,Dubai,Public Register,NaN,Dubai International Financial Centre,NaN,www.difc.ae/public-register,NaN
116,1994416,984500UAXGE4D2451072,EVIL EMPIRE NORGE LIMITED,RA000472,NaN,991 223 436,GB,ACTIVE,ISSUED,529900T8BM49AURSDO55,...,RA000472,Norway,NO,Norway,The Register of Business Enterprises,Foretaksregisteret,The Brønnøysund Register Centre,NaN,https://www.brreg.no/,NaN


In [183]:
drop_duplicates= data_merged.drop_duplicates('LEI')
drop_duplicates

,index,LEI,Entity.LegalName,Entity.RegistrationAuthority.RegistrationAuthorityID,Entity.RegistrationAuthority.OtherRegistrationAuthorityID,Entity.RegistrationAuthority.RegistrationAuthorityEntityID,Entity.LegalJurisdiction,Entity.EntityStatus,Registration.RegistrationStatus,Registration.ManagingLOU,...,Registration Authority Code,Country,Country Code,Jurisdiction (country or region),International name of Register,Local name of Register,International name of organisation responsible for the Register,Local name of organisation responsible for the Register,Website,Comments
0,12037,2138002FA7TJLH9VCN93,NDM LIMITED,RA000161,NaN,ΑΕ 3085,KY,ACTIVE,ISSUED,213800WAVVOPS85N2205,...,RA000161,Cyprus,CY,Cyprus,Companies Section,NaN,Department of Registrar of Companies and Offic...,NaN,http://www.mcit.gov.cy/mcit/drcor/drcor.nsf/in...,NaN
1,47581,2138008NLWEZC6R7KJ57,IGNIS FUNDS SICAV - IGNIS ASIA PACIFIC FUND,RA000404,NaN,612291,LU,INACTIVE,RETIRED,213800WAVVOPS85N2205,...,RA000404,Ireland,IE,Ireland,Registers for financial service providers and ...,NaN,Central Bank of Ireland,NaN,registers.centralbank.ie,NaN
2,52192,2138009H2TBPW7JVN539,CIENEGA HOLDING B.V.,RA000443,NaN,OC 419,NL,INACTIVE,RETIRED,213800WAVVOPS85N2205,...,RA000443,Malta,MT,Malta,Registry of Companies,Registru tal-Kumpaniji ta'Malta,Malta Financial Services Authority - Registry ...,NaN,https://registry.mbr.mt/ROC/index.jsp,NaN
3,65214,213800BRGVDAR6VH3561,IGNIS FUNDS SICAV - IGNIS CHINA FUND,RA000404,NaN,612991,LU,INACTIVE,RETIRED,213800WAVVOPS85N2205,...,RA000404,Ireland,IE,Ireland,Registers for financial service providers and ...,NaN,Central Bank of Ireland,NaN,registers.centralbank.ie,NaN
4,70669,213800CPQOTAW6TZOY27,HENDERSON HORIZON FUND - WORLD SELECT FUND,RA000592,NaN,144111,LU,INACTIVE,RETIRED,213800WAVVOPS85N2205,...,RA000592,United Kingdom,GB,United Kingdom,Financial Services Register,Financial Services Register,Financial Conduct Authority,Financial Conduct Authority,https://register.fca.org.uk,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113,1988944,984500G768A11O365727,EIFFAGE INFRAESTRUCTURAS SA SPÓŁKA AKCYJNA ODD...,RA000484,NaN,0000833468,ES,ACTIVE,ISSUED,529900T8BM49AURSDO55,...,RA000484,Poland,PL,Poland,National Court Register,Krajowy Rejestr Sądowy (KRS),Ministry of Justice,Minsterstwo Sprawiedliwosci,https://ekrs.ms.gov.pl/web/wyszukiwarka-krs/st...,NaN
114,1989635,984500I13E8FF4CD7C96,TEKNOVEKST,RA000472,NaN,995 476 711,GB,ACTIVE,ISSUED,529900T8BM49AURSDO55,...,RA000472,Norway,NO,Norway,The Register of Business Enterprises,Foretaksregisteret,The Brønnøysund Register Centre,NaN,https://www.brreg.no/,NaN
115,1993130,984500R2B00E5EE0DC75,Kotak Mahindra Bank Limited,RA000660,NaN,3338,IN,ACTIVE,ISSUED,529900T8BM49AURSDO55,...,RA000660,United Arab Emirates,AE,Dubai,Public Register,NaN,Dubai International Financial Centre,NaN,www.difc.ae/public-register,NaN
116,1994416,984500UAXGE4D2451072,EVIL EMPIRE NORGE LIMITED,RA000472,NaN,991 223 436,GB,ACTIVE,ISSUED,529900T8BM49AURSDO55,...,RA000472,Norway,NO,Norway,The Register of Business Enterprises,Foretaksregisteret,The Brønnøysund Register Centre,NaN,https://www.brreg.no/,NaN


In [185]:
drop_duplicates.groupby(['Entity.LegalJurisdiction', 'Country Code']).size().reset_index()

,Entity.LegalJurisdiction,Country Code,0
0,AE,SA,1
1,AT,IT,1
2,AU,NZ,1
3,BE,FR,1
4,BE,HU,1
5,CH,IN,1
6,CN,PA,1
7,CW,BE,1
8,DE,FI,1
9,DE,HU,1


In [187]:
drop_duplicates[(drop_duplicates['Entity.LegalJurisdiction'] == 'GB') & (drop_duplicates['Country Code'] == 'NO') ]

,index,LEI,Entity.LegalName,Entity.RegistrationAuthority.RegistrationAuthorityID,Entity.RegistrationAuthority.OtherRegistrationAuthorityID,Entity.RegistrationAuthority.RegistrationAuthorityEntityID,Entity.LegalJurisdiction,Entity.EntityStatus,Registration.RegistrationStatus,Registration.ManagingLOU,...,Registration Authority Code,Country,Country Code,Jurisdiction (country or region),International name of Register,Local name of Register,International name of organisation responsible for the Register,Local name of organisation responsible for the Register,Website,Comments
10,235440,2549005GBUCW6ZFKW564,PROFITUS,RA000472,NaN,893 205 942,GB,ACTIVE,ISSUED,529900T8BM49AURSDO55,...,RA000472,Norway,NO,Norway,The Register of Business Enterprises,Foretaksregisteret,The Brønnøysund Register Centre,NaN,https://www.brreg.no/,NaN
17,981934,549300J1JXHHJBD6QV25,PAAL MYHRVANG LTD,RA000472,NaN,988 095 141,GB,ACTIVE,ISSUED,529900T8BM49AURSDO55,...,RA000472,Norway,NO,Norway,The Register of Business Enterprises,Foretaksregisteret,The Brønnøysund Register Centre,NaN,https://www.brreg.no/,NaN
34,1797785,967600PHWRUW1KOHSF81,KISCON LTD,RA000472,NaN,989 843 214,GB,ACTIVE,ISSUED,529900T8BM49AURSDO55,...,RA000472,Norway,NO,Norway,The Register of Business Enterprises,Foretaksregisteret,The Brønnøysund Register Centre,NaN,https://www.brreg.no/,NaN
39,1901347,9845000F9D0C5F04FC35,DALER INN LIMITED,RA000472,NaN,993 890 693,GB,ACTIVE,ISSUED,529900T8BM49AURSDO55,...,RA000472,Norway,NO,Norway,The Register of Business Enterprises,Foretaksregisteret,The Brønnøysund Register Centre,NaN,https://www.brreg.no/,NaN
42,1904154,9845001AFC705D9DAP48,CREDIBILITY LIE DETECTION TECHNOLOGY,RA000472,NaN,921 155 107,GB,ACTIVE,ISSUED,529900T8BM49AURSDO55,...,RA000472,Norway,NO,Norway,The Register of Business Enterprises,Foretaksregisteret,The Brønnøysund Register Centre,NaN,https://www.brreg.no/,NaN
63,1919394,98450050C85715C8DF62,OBLEON,RA000472,NaN,996 528 855,GB,ACTIVE,ISSUED,529900T8BM49AURSDO55,...,RA000472,Norway,NO,Norway,The Register of Business Enterprises,Foretaksregisteret,The Brønnøysund Register Centre,NaN,https://www.brreg.no/,NaN
67,1932970,984500763DC3ADD3FF92,INDEPENDENT INVESTOR,RA000472,NaN,989 060 643,GB,ACTIVE,ISSUED,529900T8BM49AURSDO55,...,RA000472,Norway,NO,Norway,The Register of Business Enterprises,Foretaksregisteret,The Brønnøysund Register Centre,NaN,https://www.brreg.no/,NaN
71,1935427,9845007CDEE88B61VX32,DALSAUNET HOLDING,RA000472,NaN,995 664 313,GB,ACTIVE,ISSUED,529900T8BM49AURSDO55,...,RA000472,Norway,NO,Norway,The Register of Business Enterprises,Foretaksregisteret,The Brønnøysund Register Centre,NaN,https://www.brreg.no/,NaN
73,1938615,9845008593O88042AF60,HAUGEBERG&HÅVERSEN HAGE OG ANLEGGS-SERVICE,RA000472,NaN,925 447 471,GB,ACTIVE,ISSUED,529900T8BM49AURSDO55,...,RA000472,Norway,NO,Norway,The Register of Business Enterprises,Foretaksregisteret,The Brønnøysund Register Centre,NaN,https://www.brreg.no/,NaN
74,1938690,98450085BEDHA70CDF49,GEMO CAPITAL CONSULTING LIMITED,RA000472,NaN,893 136 142,GB,ACTIVE,ISSUED,529900T8BM49AURSDO55,...,RA000472,Norway,NO,Norway,The Register of Business Enterprises,Foretaksregisteret,The Brønnøysund Register Centre,NaN,https://www.brreg.no/,NaN


### Results as a single excel file

In [169]:
results = pd.ExcelWriter('D:/SVL/Chunks/findingAnomalies.xlsx')
new_chunks.to_excel(results, sheet_name = 'Anomalies')
results.close()
